In [1]:
import numpy as np
import pandas as pd
import os
import cv2
import time
from ultralytics import YOLO

In [2]:
path = "archive/YOLO_format/train"

print("Contoh isi folder gambar:", os.listdir(f"{path}/images")[:5])
print("Contoh isi folder label :", os.listdir(f"{path}/labels")[:5])

Contoh isi folder gambar: ['ffhq_0.png', 'ffhq_1.png', 'ffhq_10.png', 'ffhq_100.png', 'ffhq_1000.png']
Contoh isi folder label : ['ffhq_0.txt', 'ffhq_1.txt', 'ffhq_10.txt', 'ffhq_100.txt', 'ffhq_1000.txt']


In [3]:
yaml_content = """
path: archive/YOLO_format/
train: train/images
val: valid/images
test: test/images

nc: 8
names: ['neutral', 'happy', 'sad', 'surprise', 'fear', 'disgust', 'anger', 'contempt']
"""
with open("affectnet.yaml", "w") as f:
    f.write(yaml_content)

In [4]:
model = YOLO("yolov8n.pt")
results = model.train(
    data="affectnet.yaml",
    epochs=20,
    imgsz=224,
    batch=32,
    project="runs/detect",
    name="train",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.240 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.232  Python-3.11.9 torch-2.9.1+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=affectnet.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, op

In [8]:
model = YOLO("runs/detect/train/weights/best.pt")
results = model.val(data="affectnet.yaml")
print("Metrik Evaluasi:", results.box.map)

Ultralytics 8.3.232  Python-3.11.9 torch-2.9.1+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 0.70.5 MB/s, size: 11.0 KB)
val: Scanning C:\Users\Jeremy\Documents\Kuliah\ML-AI\UAS LAB\archive\YOLO_format\valid\labels.cache... 5406 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5406/5406 1.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 338/338 1.4it/s 3:58<0.7s
                   all       5406       5406      0.717      0.755      0.815      0.803
               neutral        712        712      0.709      0.747      0.802      0.783
                 happy        618        618      0.685      0.812      0.836      0.833
                   sad        672        672      0.627      0.754      0.774      0.768
              surprise        622        622      0.766      0.742      0.

In [9]:
model.export(format="onnx")

Ultralytics 8.3.232  Python-3.11.9 torch-2.9.1+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)

PyTorch: starting from 'runs\detect\train\weights\best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 12, 1029) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<=1.19.1', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--

'runs\\detect\\train\\weights\\best.onnx'

In [ ]:
import cv2
from ultralytics import YOLO
import time

model = YOLO('runs/detect/train/weights/best.pt')

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Username atau password RTSP salah")
    exit()

prev_time = time.time()

while True:
    success, frame = cap.read()

    if not success or frame is None:
        print("Gagal membaca frame. coba lagi...")
        cap.release()
        time.sleep(1)
        cap = cv2.VideoCapture(0)
        continue

    # Hitung FPS
    current_time = time.time()
    fps = 1 / (current_time - prev_time)
    prev_time = current_time

    # YOLO inference
    results = model(frame)
    annotated_frame = results[0].plot()

    # Tambahkan FPS ke video
    cv2.putText(annotated_frame, f"FPS: {int(fps)}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    # Tampilkan hasil
    cv2.imshow("YOLO RTSP CCTV Inference", annotated_frame)

    # Tekan 'q' untuk keluar
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()